# Gemma 4 Developer Agent — Local Run (llama.cpp / GGUF)

Local port of the Kaggle starter notebook, adapted to serve a **q4_0 GGUF via llama.cpp** so it fits a single 8 GB GPU (e.g. RTX 5060, Blackwell sm_120).

**Environment (already set up in `.venv`):**
- Python 3.12 venv built with `uv`.
- `torch==2.10.0+cu128` (Blackwell sm_120 support) + the wheelhouse + missing PyPI deps.
- Prebuilt `llama.cpp` CUDA 12.8 binaries under `llama.cpp/llama-b11972/`.
- The vLLM cell is replaced by a llama.cpp launcher; cells 0–3 and 5–6 are unchanged.

## 0. Local Configuration (EDIT THIS CELL)

In [ ]:
from pathlib import Path

# =============================================================================
# EDIT THESE PATHS FOR YOUR MACHINE
# =============================================================================

# Root of this project (folder that contains 'gemma-4-developer-agent' and the wheelhouse).
# In a notebook there is no __file__, so set this explicitly on the GPU box, e.g.:
#   PROJECT_ROOT = Path('/home/you/SWE-Agent')
# By default we auto-detect by walking up from the kernel's CWD looking for the data folder.
def _find_project_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / 'gemma-4-developer-agent').is_dir() and (cand / 'gemma-4-developer-agent-wheelhouse').is_dir():
            return cand
    return start

PROJECT_ROOT = _find_project_root(Path.cwd().resolve())

# The competition DATA directory (contains tasks.jsonl, snapshots/, graphs/, embeddings/, wheels/, sandbox/, sample_submission/).
DATA_DIR = PROJECT_ROOT / 'gemma-4-developer-agent'

# The wheelhouse folder with all the .whl files (swegemma, adk_submission, vllm, torch, ...).
WHEELHOUSE_DIR = PROJECT_ROOT / 'gemma-4-developer-agent-wheelhouse'

# A writable scratch dir (replaces /kaggle/working).
WORKING_DIR = PROJECT_ROOT / 'working'

# -----------------------------------------------------------------------------
# MODEL
# -----------------------------------------------------------------------------
# Which competition model to run. Smallest-first (fits a single consumer GPU best):
#   'gemma-4-e2b-it-qat-w4a16-ct'   (smallest, ~2-3 GB weights)   <- default, closest to a small local model
#   'gemma-4-e4b-it-qat-w4a16-ct'   (~3-4 GB)
#   'gemma-4-12b-it-qat-w4a16-ct'   (~6-8 GB)
#   'gemma-4-31b-it-qat-w4a16-ct'   (~16-18 GB, needs a big GPU / multi-GPU)
# On an 8 GB RTX 5060 use a 4-bit (W4A16) E4B checkpoint; unquantized E4B (~16 GB bf16) will not fit.
MODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'

# Local directory that holds the downloaded model weights for MODEL_NAME.
# Point this at wherever you download/unpack the model (an HF-format folder with
# config.json + *.safetensors + tokenizer files). Example layouts:
#   PROJECT_ROOT / 'models' / MODEL_NAME
#   Path('/home/you/models') / MODEL_NAME
MODEL_PATH = PROJECT_ROOT / 'models' / MODEL_NAME

# vLLM serving knobs for a single-GPU box. Lower gpu_memory_utilization if you hit OOM;
# lower MAX_MODEL_LEN (e.g. 8192) to shrink the KV cache on small-VRAM GPUs.
GPU_MEMORY_UTILIZATION = 0.90  # (unused on the llama.cpp path)
MAX_MODEL_LEN = 16384  # llama.cpp --ctx-size; fits 8 GB VRAM with flash-attn (~4.3 GB used)

# How many sample tasks to actually evaluate (keep small for a first run).
NUM_SAMPLE_TASKS = 1


# -----------------------------------------------------------------------------
# LLAMA.CPP SERVER (GGUF path for small GPUs)
# -----------------------------------------------------------------------------
# MODEL_PATH should point at a folder containing a *.gguf (or directly at the .gguf).
# Download e.g. google/gemma-4-E4B-it-qat-q4_0-gguf (file: gemma-4-E4B_q4_0-it.gguf).
# Path to the llama-server binary from the prebuilt CUDA 12.8 release.
LLAMA_SERVER_BIN = PROJECT_ROOT / 'llama.cpp' / 'llama-b11972' / 'llama-server'
# -1 or a big number offloads all layers to the GPU; lower it if you OOM.
LLAMA_N_GPU_LAYERS = 999
# Seconds to wait for the server to become healthy.
LLAMA_STARTUP_TIMEOUT = 600
print('LLAMA_SERVER_BIN =', LLAMA_SERVER_BIN, '(exists:', LLAMA_SERVER_BIN.exists(), ')')
# =============================================================================
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT   =', PROJECT_ROOT)
print('DATA_DIR       =', DATA_DIR, '(exists:', DATA_DIR.exists(), ')')
print('WHEELHOUSE_DIR =', WHEELHOUSE_DIR, '(exists:', WHEELHOUSE_DIR.exists(), ')')
print('WORKING_DIR    =', WORKING_DIR)
print('MODEL_NAME     =', MODEL_NAME)
print('MODEL_PATH     =', MODEL_PATH, '(exists:', MODEL_PATH.exists(), ')')

## 1. Environment Configuration and Package Installation

Installs the harness and all dependencies from the **local wheelhouse** (offline, no PyPI).

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'
# NOTE: we intentionally do NOT set TRANSFORMERS_OFFLINE=1 here so you can download the
# model the first time if needed. Set it to '1' once weights are local to force offline.


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM', 'LiteLLM Router', 'LiteLLM Proxy', 'litellm',
        'httpx', 'httpcore', 'openai', 'urllib3',
        'opentelemetry', 'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

assert WHEELHOUSE_DIR.exists(), f'Wheelhouse not found at {WHEELHOUSE_DIR}'

# Install every wheel in the local wheelhouse. --no-deps + --no-index keeps it fully offline;
# the wheelhouse is expected to be self-contained (all transitive deps included).
wheels = sorted(str(w) for w in WHEELHOUSE_DIR.glob('*.whl') if 'cutlass' not in w.name.lower())
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
     f'--find-links={WHEELHOUSE_DIR}', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Dataset and Sample Submission

In [ ]:
import re
from swegemma.models import load_tasks

assert DATA_DIR.exists(), f'DATA_DIR not found: {DATA_DIR}'

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# Rewrite the agent YAMLs to the chosen MODEL_NAME. The sample LoRA adapters are shaped for
# the 31B model, so when we pick a different base model we strip the adapters and the
# 'adapter:' lines (otherwise vLLM would reject the mismatched adapter shapes).
if MODEL_NAME != 'gemma-4-31b-it-qat-w4a16-ct':
    for yaml_path in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
        if yaml_path.is_file():
            text = yaml_path.read_text(encoding='utf-8')
            text = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', text, flags=re.MULTILINE)
            text = re.sub(r'^\s*adapter:\s*.*\n?', '', text, flags=re.MULTILINE)
            yaml_path.write_text(text, encoding='utf-8')
    shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Load tasks from the local dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Code Graph Functions (CPU-only, no model needed)

This cell exercises the pre-computed repo code graph + embeddings. It needs no GPU, so it's a
good first check that the harness imported and the data paths are correct.

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(node=query_node, graph=repo_graph, max_neighbors=10)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

similar = sg.get_similar_nodes(
    node=query_node, repo_name=sample_task.repo, k=5, graph=repo_graph,
    graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start llama.cpp Server (GGUF, needs GPU)

Serves the q4_0 GGUF on an OpenAI-compatible endpoint at `127.0.0.1:8000` with `--jinja` (so Gemma 4 tool calls parse into standard `tool_calls`), then builds the ADK model registry pointing at it. This replaces the vLLM path, whose full-precision weights do not fit an 8 GB GPU for the Gemma 4 E-models.

In [ ]:
# 4. Start llama.cpp server (GGUF, fits small GPUs) + build the ADK model registry.
#
# The bundled vLLM path needs the full bf16/W4A16 weights resident in VRAM. The Gemma 4
# E-models are multimodal, so even the "small" E2B W4A16 checkpoint is ~8 GB and the E4B
# is ~11.5 GB -- neither fits an 8 GB card. Instead we serve a q4_0 GGUF with llama.cpp
# (E4B q4_0 ~5.1 GB, E2B q4_0 ~3.4 GB) and expose the same OpenAI-compatible /v1 API on
# 127.0.0.1:8000 that the harness expects, then build the ModelRegistry by hand exactly
# like VllmServer.create_model_registry does.
import socket
import subprocess
import time
import urllib.request

import litellm
from adk_submission import ModelRegistry, discover_adapters
from adk_submission.discovery import discover_declared_models
from adk_submission.resolvers.generation import install_litellm_reasoning_patch
from google.adk.models.lite_llm import LiteLlm
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True
install_litellm_reasoning_patch()

INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')
LLAMA_HOST = '127.0.0.1'
LLAMA_PORT = 8000
LLAMA_BASE_URL = f'http://{LLAMA_HOST}:{LLAMA_PORT}/v1'

# Validate the submission still declares exactly one model, and use it as the served alias.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))

# llama.cpp loads GGUF, so MODEL_PATH must point at a .gguf file (not an HF folder).
gguf_path = MODEL_PATH
if gguf_path.is_dir():
    ggufs = sorted(p for p in gguf_path.glob('*.gguf') if 'mmproj' not in p.name.lower())
    assert ggufs, f'No non-mmproj *.gguf found under {gguf_path}'
    gguf_path = ggufs[0]
assert gguf_path.is_file(), f'GGUF weights not found at {gguf_path}'

# Adapters are not used on the GGUF/llama.cpp path (cell 2 strips them for non-31B models).
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)


def _port_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0


def _health_ok(url, timeout=1.0):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False


# Reuse a server we started earlier in this kernel; otherwise launch one.
llama_proc = globals().get('llama_proc')
health_url = f'http://{LLAMA_HOST}:{LLAMA_PORT}/health'

if _health_ok(health_url):
    print(f'Reusing llama.cpp server already listening on {LLAMA_BASE_URL}')
else:
    assert not _port_open(LLAMA_HOST, LLAMA_PORT), (
        f'Port {LLAMA_PORT} is busy but /health is not OK. Stop whatever is on it and re-run.'
    )
    llama_bin = LLAMA_SERVER_BIN
    assert llama_bin.is_file(), (
        f'llama-server not found at {llama_bin}. Set LLAMA_SERVER_BIN in the CONFIG cell.'
    )
    env = dict(os.environ)
    env['LD_LIBRARY_PATH'] = f'{llama_bin.parent}:' + env.get('LD_LIBRARY_PATH', '')
    cmd = [
        str(llama_bin),
        '--model', str(gguf_path),
        '--host', LLAMA_HOST, '--port', str(LLAMA_PORT),
        '--alias', declared_model,
        '--n-gpu-layers', str(LLAMA_N_GPU_LAYERS),
        '--ctx-size', str(MAX_MODEL_LEN),
        '--jinja',                 # use Gemma 4's chat template so tool calls parse to OpenAI tool_calls
        '--flash-attn', 'on',
        '--parallel', '1',
    ]
    print('Launching:', ' '.join(cmd))
    llama_log = open(WORKING_DIR / 'llama_server.log', 'w')
    llama_proc = subprocess.Popen(cmd, env=env, stdout=llama_log, stderr=subprocess.STDOUT)
    globals()['llama_proc'] = llama_proc

    deadline = time.time() + LLAMA_STARTUP_TIMEOUT
    while time.time() < deadline:
        if llama_proc.poll() is not None:
            raise RuntimeError(
                f'llama-server exited early (code {llama_proc.returncode}). '
                f'See {WORKING_DIR / "llama_server.log"}'
            )
        if _health_ok(health_url):
            break
        time.sleep(2)
    else:
        raise TimeoutError(f'llama-server not healthy after {LLAMA_STARTUP_TIMEOUT}s')
    print(f'llama.cpp server healthy on {LLAMA_BASE_URL} (model={declared_model})')

# Build the registry the same way VllmServer.create_model_registry does:
# LiteLlm(model="openai/<alias>", api_base=..., api_key=...). We also pin
# custom_llm_provider="openai" so a bare request model still routes to this endpoint.
models = ModelRegistry()
models.register(
    declared_model,
    LiteLlm(
        model=f'openai/{declared_model}',
        api_base=LLAMA_BASE_URL,
        api_key=INFERENCE_API_KEY,
        custom_llm_provider='openai',
    ),
)
print('Model registry ready:', declared_model, '->', LLAMA_BASE_URL)


## 5. Run Phase 1 Inference and Phase 2 Verification

Uses the `subprocess` sandbox backend (no Docker needed).

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := AGENT_DIR / name).is_file()),
    None,
)
raw_eval_cfg = (
    yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}
    if eval_config_file is not None else {}
)
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

SAMPLE_TASKS = tasks[:NUM_SAMPLE_TASKS]
limits, gen_constraints = build_submission_limits()

eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    result = run_sync(
        evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SAMPLE_TASKS),
    )
    patch = (result.agent_patch or '').strip()
    predictions.append({'id': task.instance_id, 'prediction': patch if patch else 'NO_PATCH'})
    print(
        f'  -> resolved={result.resolved}, exit_code={result.test_exit_code}, '
        f'patch_chars={len(patch)}, tool_calls={result.tool_calls}, '
        f'duration={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
display(submission_df)

## 6. Package Submission Archive

In [ ]:
import zipfile
from adk_submission import validate_directory
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

for dirpath, dirnames, filenames in os.walk(AGENT_DIR, followlinks=False):
    parent = Path(dirpath)
    for name in (*dirnames, *filenames):
        assert not (parent / name).is_symlink(), f'Symlinks are not permitted: {(parent / name).relative_to(AGENT_DIR)}'
validate_directory(AGENT_DIR, limits=limits)

zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')